# 02 — Pareto Plots

Compute and visualise Pareto frontiers over (cost, accuracy) space for the
full experiment results.
Run `python experiments/02_full_run.py` first.

## Setup

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='colorblind')

RESULTS_DIR = Path('../results/full')
BUDGET_TIER = 'mid'  # change to 'low' or 'high' as needed

def is_pareto(costs, scores):
    """Return boolean mask of Pareto-optimal points (min cost, max score)."""
    n = len(costs)
    dominated = np.zeros(n, dtype=bool)
    for i in range(n):
        for j in range(n):
            if i != j:
                if costs[j] <= costs[i] and scores[j] >= scores[i]:
                    if costs[j] < costs[i] or scores[j] > scores[i]:
                        dominated[i] = True
                        break
    return ~dominated

## Load Full Results

In [ ]:
summaries = []
for path in sorted((RESULTS_DIR / BUDGET_TIER).glob('*_summary.json')):
    summaries.append(json.loads(path.read_text()))

df = pd.DataFrame(summaries)
print(f"Loaded {len(df)} runs across {df['dataset'].nunique()} datasets")

# Aggregate across seeds.
agg = df.groupby(['strategy', 'dataset']).agg(
    mean_sr=('success_rate', 'mean'),
    std_sr=('success_rate', 'std'),
    mean_cost=('total_cost_usd', 'mean'),
).reset_index()
agg.head(10)

## Per-Dataset Pareto Frontier

In [ ]:
datasets = agg['dataset'].unique()
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

for ax, dataset in zip(axes.flat, datasets):
    sub = agg[agg['dataset'] == dataset].copy()
    mask = is_pareto(sub['mean_cost'].values, sub['mean_sr'].values)
    frontier = sub[mask].sort_values('mean_cost')

    for _, row in sub.iterrows():
        ax.scatter(row['mean_cost'], row['mean_sr'],
                   label=row['strategy'], s=70, zorder=3)
    if len(frontier) > 1:
        ax.plot(frontier['mean_cost'], frontier['mean_sr'],
                'k--', linewidth=1, alpha=0.5, label='Pareto frontier')
    ax.set_title(dataset)
    ax.set_xlabel('Mean cost (USD)')
    ax.set_ylabel('Mean success rate')
    ax.legend(fontsize=7)

plt.suptitle(f'Per-Dataset Pareto Frontiers — {BUDGET_TIER} budget', fontsize=13)
plt.tight_layout()
plt.show()

## Cross-Dataset Comparison

In [ ]:
cross = agg.pivot_table(
    index='strategy', columns='dataset', values='mean_sr'
)
fig, ax = plt.subplots(figsize=(9, 4))
sns.heatmap(cross, annot=True, fmt='.3f', cmap='RdYlGn', center=0.5, ax=ax)
ax.set_title('Mean Success Rate — strategy × dataset')
plt.tight_layout()
plt.show()

## Heterogeneity Ablation

Compare `vanilla_debate` (homogeneous) vs. `hetero_debate` (mixed models)
at matched cost to isolate the effect of model diversity.

In [ ]:
ablation = agg[agg['strategy'].isin(['vanilla_debate', 'hetero_debate'])].copy()
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=ablation, x='dataset', y='mean_sr', hue='strategy', ax=ax)
ax.set_title('Heterogeneity Ablation: vanilla vs. hetero debate')
ax.set_ylabel('Mean success rate')
plt.tight_layout()
plt.show()